# Mango Leaf Disease Detection — Inception V3 Transfer Learning

This notebook trains the two models used by the production backend:

1. **Disease classifier** — Inception V3 (ImageNet-pretrained, transfer learning) fine-tuned on the
   [Mango Leaf Disease Dataset](https://www.kaggle.com/datasets/aryashah2k/mango-leaf-disease-dataset) (Kaggle).
2. **Leaf gate** — a small binary classifier (MobileNetV2 backbone) that answers "is this even a
   mango leaf?" before the disease classifier is trusted. This exists because a softmax classifier
   trained only on mango leaf disease classes will happily assign a confident-looking label to a
   photo of a dog, a car, or a different plant's leaf — it was never given the option to say "none
   of these."

Outputs of this notebook, copied into `backend/app/ml/`:

| File | Produced by |
|---|---|
| `model.tflite` | Disease classifier, converted from Keras `.h5` |
| `leaf_gate.tflite` | Binary gate classifier |
| `labels.json` | Class index → disease name mapping |

**Why TFLite over ONNX for serving:** the backend is deployed on Render's free tier, which has a
tight memory ceiling. A full TensorFlow/Keras runtime in the serving container was previously found
to OOM-crash under load; TFLite's interpreter has a much smaller runtime footprint and loads only
the quantized/converted graph, not the full training framework. ONNX export code is included below
as a documented alternative in case the model needs to move to an ONNX Runtime-based host later.

## 1. Environment setup

In [ ]:
!pip install -q tensorflow==2.17.1 tensorflow-datasets scikit-learn matplotlib seaborn kagglehub tf2onnx onnx pillow

In [ ]:
import os
import json
import shutil
import random
import pathlib

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input as inception_preprocess
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input as mobilenet_preprocess
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))

## 2. Download the dataset

Uses `kagglehub`, which works in both Colab and Kaggle notebooks without manual `kaggle.json`
juggling (on Kaggle it authenticates automatically; on Colab it will prompt for Kaggle credentials
once).

In [ ]:
import kagglehub

DATASET_HANDLE = "aryashah2k/mango-leaf-disease-dataset"
dataset_root = kagglehub.dataset_download(DATASET_HANDLE)
print("Dataset downloaded to:", dataset_root)

# The Kaggle dataset ships as MangoLeafBD Dataset/<ClassName>/*.jpg
data_dir = next(p for p in pathlib.Path(dataset_root).rglob("*") if p.is_dir() and any(c.is_dir() for c in p.iterdir()))
class_dirs = sorted([d for d in data_dir.iterdir() if d.is_dir()])
CLASS_NAMES = [d.name for d in class_dirs]
print(f"Found {len(CLASS_NAMES)} classes:", CLASS_NAMES)

## 3. Exploratory look at class balance

In [ ]:
counts = {d.name: len(list(d.glob("*"))) for d in class_dirs}
plt.figure(figsize=(10, 4))
sns.barplot(x=list(counts.keys()), y=list(counts.values()))
plt.xticks(rotation=45, ha="right")
plt.ylabel("Image count")
plt.title("Images per class")
plt.tight_layout()
plt.show()
print(counts)

## 4. Data pipelines

70/15/15 train/val/test split, InceptionV3's expected 299x299 input, light augmentation on the
training set only (mirrors real-world capture variance — phone camera angle, lighting) without
distorting the disease-relevant leaf texture (no aggressive color jitter).

In [ ]:
IMG_SIZE = (299, 299)
BATCH_SIZE = 32

raw_train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.3, subset="training", seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="categorical", class_names=CLASS_NAMES,
)
raw_val_test_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.3, subset="validation", seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="categorical", class_names=CLASS_NAMES,
)

# Split the 30% holdout into equal val/test halves
val_batches = tf.data.experimental.cardinality(raw_val_test_ds)
raw_val_ds = raw_val_test_ds.take(val_batches // 2)
raw_test_ds = raw_val_test_ds.skip(val_batches // 2)

augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

def prep_train(x, y):
    x = augment(x)
    return inception_preprocess(x), y

def prep_eval(x, y):
    return inception_preprocess(x), y

AUTOTUNE = tf.data.AUTOTUNE
train_ds = raw_train_ds.map(prep_train, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds = raw_val_ds.map(prep_eval, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_ds = raw_test_ds.map(prep_eval, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

NUM_CLASSES = len(CLASS_NAMES)
print("Classes:", NUM_CLASSES)

## 5. Disease classifier — Inception V3 transfer learning

Two-phase training:
1. **Head-only** — freeze the Inception V3 backbone entirely, train only the new classification
   head with a relatively high learning rate until it converges.
2. **Fine-tune** — unfreeze the top block of the backbone and continue training at a much lower
   learning rate, so the pretrained low/mid-level filters adapt slightly to leaf texture without
   catastrophically forgetting ImageNet features.

In [ ]:
base_model = InceptionV3(weights="imagenet", include_top=False, input_shape=(*IMG_SIZE, 3))
base_model.trainable = False

inputs = layers.Input(shape=(*IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = models.Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

In [ ]:
early_stop = callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True)
reduce_lr = callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6)
checkpoint = callbacks.ModelCheckpoint("best_head.h5", monitor="val_accuracy", save_best_only=True)

history_head = model.fit(
    train_ds, validation_data=val_ds, epochs=15,
    callbacks=[early_stop, reduce_lr, checkpoint],
)

In [ ]:
# Phase 2: fine-tune the top of the backbone
base_model.trainable = True
FINE_TUNE_FROM = len(base_model.layers) - 50  # unfreeze last ~50 layers
for layer in base_model.layers[:FINE_TUNE_FROM]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

checkpoint_ft = callbacks.ModelCheckpoint("best_finetuned.h5", monitor="val_accuracy", save_best_only=True)

history_finetune = model.fit(
    train_ds, validation_data=val_ds, epochs=10,
    callbacks=[early_stop, reduce_lr, checkpoint_ft],
)

## 6. Training curves

In [ ]:
def plot_history(histories, labels):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for h, label in zip(histories, labels):
        axes[0].plot(h.history["accuracy"], label=f"{label} train")
        axes[0].plot(h.history["val_accuracy"], label=f"{label} val", linestyle="--")
        axes[1].plot(h.history["loss"], label=f"{label} train")
        axes[1].plot(h.history["val_loss"], label=f"{label} val", linestyle="--")
    axes[0].set_title("Accuracy"); axes[0].set_xlabel("Epoch"); axes[0].legend()
    axes[1].set_title("Loss"); axes[1].set_xlabel("Epoch"); axes[1].legend()
    plt.tight_layout()
    plt.show()

plot_history([history_head, history_finetune], ["head", "fine-tune"])

## 7. Evaluation — confusion matrix and classification report

In [ ]:
y_true, y_pred, y_conf = [], [], []
for images, labels_batch in test_ds:
    probs = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels_batch.numpy(), axis=1))
    y_pred.extend(np.argmax(probs, axis=1))
    y_conf.extend(np.max(probs, axis=1))

y_true, y_pred, y_conf = np.array(y_true), np.array(y_pred), np.array(y_conf)

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion matrix — disease classifier")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

test_accuracy = (y_true == y_pred).mean()
print(f"Test accuracy: {test_accuracy:.4f}")

## 8. Confidence threshold for the disease classifier

Out-of-distribution inputs (and genuinely ambiguous/borderline photos) tend to produce a flatter,
less peaked softmax distribution than a confident, in-distribution prediction. We sweep candidate
thresholds over the **validation** set (not test, to avoid leaking the threshold decision into the
reported test metrics) and pick the value that keeps most correct predictions while rejecting a
meaningful share of the classifier's mistakes.

In [ ]:
val_true, val_pred, val_conf = [], [], []
for images, labels_batch in val_ds:
    probs = model.predict(images, verbose=0)
    val_true.extend(np.argmax(labels_batch.numpy(), axis=1))
    val_pred.extend(np.argmax(probs, axis=1))
    val_conf.extend(np.max(probs, axis=1))

val_true, val_pred, val_conf = np.array(val_true), np.array(val_pred), np.array(val_conf)
val_correct = (val_true == val_pred)

candidate_thresholds = np.arange(0.50, 0.96, 0.05)
print(f"{'threshold':>10} {'kept_frac':>10} {'accuracy_on_kept':>18}")
for t in candidate_thresholds:
    kept = val_conf >= t
    kept_frac = kept.mean()
    acc_on_kept = val_correct[kept].mean() if kept.any() else float("nan")
    print(f"{t:>10.2f} {kept_frac:>10.2%} {acc_on_kept:>18.2%}")

# Chosen operating point: documented after inspecting the table above. 0.65 is a reasonable
# starting point for this dataset/model — keeps the large majority of correct predictions while
# cutting off the long tail of low-confidence, frequently-wrong predictions. Re-run this cell
# after any retrain and adjust CONFIDENCE_THRESHOLD if the table shifts.
CONFIDENCE_THRESHOLD = 0.65
print(f"\nSelected CONFIDENCE_THRESHOLD = {CONFIDENCE_THRESHOLD}")

## 9. Export labels and disease classifier

`labels.json` maps the softmax output index to a human-readable class name — the backend loads
this at startup so the class list never has to be hardcoded in Python.

In [ ]:
ML_DIR = pathlib.Path("../backend/app/ml")
ML_DIR.mkdir(parents=True, exist_ok=True)

labels = {str(i): name for i, name in enumerate(CLASS_NAMES)}
with open(ML_DIR / "labels.json", "w") as f:
    json.dump({
        "labels": labels,
        "confidence_threshold": CONFIDENCE_THRESHOLD,
    }, f, indent=2)

print(json.dumps(labels, indent=2))

In [ ]:
# Reference copy: full Keras model
model.save("mango_inception_v3.h5")

# --- Serving export: TFLite (chosen — see rationale in the intro cell) ---
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # dynamic-range quantization: smaller + faster, no calibration set needed
tflite_model = converter.convert()

with open(ML_DIR / "model.tflite", "wb") as f:
    f.write(tflite_model)

print("model.tflite size (MB):", len(tflite_model) / 1e6)

### Alternative: ONNX export

Not used for the deployed backend (see rationale above), but included for completeness in case the
serving target changes to an ONNX Runtime-based host.

```python
import tf2onnx

spec = (tf.TensorSpec((None, *IMG_SIZE, 3), tf.float32, name="input"),)
model_proto, _ = tf2onnx.convert.from_keras(model, input_signature=spec, opset=13,
                                             output_path=str(ML_DIR / "model.onnx"))
```

## 10. Leaf gate — binary "is this a mango leaf?" classifier

**Positive class:** every image in the mango leaf disease dataset (all classes pooled — disease
label doesn't matter here, only "is a mango leaf").

**Negative class:** a mix of (a) other plant species' leaves, so the gate learns "mango leaf"
specifically rather than just "leaf-shaped green thing", and (b) generic non-leaf photos (objects,
scenes, animals), so it also rejects completely unrelated uploads. Both are pulled from
`tensorflow_datasets` so this cell is fully reproducible without extra manual downloads:

- Other-plant leaves: `tf_flowers` (flower photos — different plant, still organic/green/textured,
  a genuinely hard negative)
- Random objects/scenes: `cats_vs_dogs` (animals, indoor/outdoor scenes — easy negatives that
  stand in for "not a plant at all")

In [ ]:
import tensorflow_datasets as tfds

GATE_IMG_SIZE = (224, 224)  # MobileNetV2 default
GATE_BATCH_SIZE = 32

# Positive examples: reuse the mango leaf directory, ignore disease sub-labels
positive_paths = [str(p) for d in class_dirs for p in d.glob("*")]
random.shuffle(positive_paths)

# Negative examples from tfds (streamed, capped to roughly match positive count)
n_target = len(positive_paths)

flowers_ds = tfds.load("tf_flowers", split="train", as_supervised=True)
catsdogs_ds = tfds.load("cats_vs_dogs", split="train", as_supervised=True)

def collect_images(ds, n, size):
    out = []
    for img, _ in ds.take(n):
        img = tf.image.resize(img, size)
        out.append(img.numpy().astype("uint8"))
    return out

neg_flowers = collect_images(flowers_ds, n_target // 2, GATE_IMG_SIZE)
neg_objects = collect_images(catsdogs_ds, n_target // 2, GATE_IMG_SIZE)
negative_images = neg_flowers + neg_objects

print(f"Positives: {len(positive_paths)}  |  Negatives: {len(negative_images)} "
      f"({len(neg_flowers)} other-plant + {len(neg_objects)} generic-object)")

In [ ]:
def load_positive(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, GATE_IMG_SIZE)
    return img

positive_images = np.stack([load_positive(p).numpy().astype("uint8") for p in positive_paths])
negative_images = np.stack(negative_images)

X = np.concatenate([positive_images, negative_images], axis=0)
y = np.concatenate([np.ones(len(positive_images)), np.zeros(len(negative_images))])

idx = np.arange(len(X))
np.random.shuffle(idx)
X, y = X[idx], y[idx]

n_val = int(0.15 * len(X))
n_test = int(0.15 * len(X))
X_test, y_test = X[:n_test], y[:n_test]
X_val, y_val = X[n_test:n_test + n_val], y[n_test:n_test + n_val]
X_train, y_train = X[n_test + n_val:], y[n_test + n_val:]

print(f"Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}")

In [ ]:
gate_base = MobileNetV2(weights="imagenet", include_top=False, input_shape=(*GATE_IMG_SIZE, 3))
gate_base.trainable = False

gate_inputs = layers.Input(shape=(*GATE_IMG_SIZE, 3))
x = mobilenet_preprocess(gate_inputs)
x = gate_base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
gate_outputs = layers.Dense(1, activation="sigmoid")(x)

gate_model = models.Model(gate_inputs, gate_outputs)
gate_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=["accuracy"])

gate_early_stop = callbacks.EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True)

gate_history = gate_model.fit(
    X_train, y_train, validation_data=(X_val, y_val),
    epochs=12, batch_size=GATE_BATCH_SIZE, callbacks=[gate_early_stop],
)

## 11. Leaf gate evaluation

In [ ]:
gate_probs = gate_model.predict(X_test, verbose=0).ravel()
gate_preds = (gate_probs >= 0.5).astype(int)

gate_test_accuracy = (gate_preds == y_test).mean()
print(classification_report(y_test, gate_preds, target_names=["not_mango_leaf", "mango_leaf"]))
print(f"Leaf gate test accuracy: {gate_test_accuracy:.4f}")

gate_cm = confusion_matrix(y_test, gate_preds)
plt.figure(figsize=(4, 4))
sns.heatmap(gate_cm, annot=True, fmt="d",
            xticklabels=["not_mango_leaf", "mango_leaf"], yticklabels=["not_mango_leaf", "mango_leaf"],
            cmap="Greens")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion matrix — leaf gate")
plt.tight_layout()
plt.show()

GATE_THRESHOLD = 0.5  # standard sigmoid decision boundary; the class balance above is ~50/50 so no skew correction needed

In [ ]:
converter_gate = tf.lite.TFLiteConverter.from_keras_model(gate_model)
converter_gate.optimizations = [tf.lite.Optimize.DEFAULT]
gate_tflite_model = converter_gate.convert()

with open(ML_DIR / "leaf_gate.tflite", "wb") as f:
    f.write(gate_tflite_model)

print("leaf_gate.tflite size (MB):", len(gate_tflite_model) / 1e6)

# Record both thresholds + the gate's own accuracy alongside the labels so the backend and this
# notebook never drift out of sync with each other.
with open(ML_DIR / "labels.json", "r") as f:
    labels_payload = json.load(f)

labels_payload["gate_threshold"] = GATE_THRESHOLD
labels_payload["gate_test_accuracy"] = float(gate_test_accuracy)
labels_payload["disease_test_accuracy"] = float(test_accuracy)

with open(ML_DIR / "labels.json", "w") as f:
    json.dump(labels_payload, f, indent=2)

print(json.dumps(labels_payload, indent=2))

## 12. Summary

| Model | File | Purpose | Threshold | Test accuracy |
|---|---|---|---|---|
| Leaf gate (MobileNetV2) | `leaf_gate.tflite` | Reject non-mango-leaf images before disease inference | sigmoid ≥ `GATE_THRESHOLD` (0.5) | see `gate_test_accuracy` in `labels.json` |
| Disease classifier (Inception V3) | `model.tflite` | Predict disease class among mango leaf classes | softmax top-1 ≥ `CONFIDENCE_THRESHOLD` (0.65) | see `disease_test_accuracy` in `labels.json` |

**Inference order enforced by the backend (`backend/app/services/inference.py`):**
1. Run the leaf gate. If its output is below `GATE_THRESHOLD`, reject immediately — never call the
   disease classifier.
2. Otherwise run the Inception V3 disease classifier. If its top-1 softmax probability is below
   `CONFIDENCE_THRESHOLD`, reject as "not confident" rather than returning a low-confidence label.
3. Otherwise return the predicted class name (from `labels.json`) and confidence score.

Both thresholds and both models' measured test accuracy are written into `labels.json` so the
backend and this notebook stay in sync — re-run this notebook after any retrain and re-copy the
three files (`model.tflite`, `leaf_gate.tflite`, `labels.json`) into `backend/app/ml/`.